# SABR Paper Reproduction Walkthrough

This notebook organizes the current project into a single step-by-step workflow so the paper replication can be run one cell at a time.

Paper:
- Choi, Hu, Kwok, *Efficient and accurate simulation of the stochastic-alpha-beta-rho model*

Project scope:
- core SABR Monte Carlo simulator
- sanity checks for limiting cases and martingale behavior
- paper-style tables and figure datasets
- quick and paper-scale validation hooks


## Contents

1. [Environment and Imports](#1.-Environment-and-Imports)
2. [Project API Overview](#2.-Project-API-Overview)
3. [Core Mathematical Formulas](#3.-Core-Mathematical-Formulas)
4. [Table 3 Cases](#4.-Table-3-Cases)
5. [Sanity Checks](#5.-Sanity-Checks)
6. [Paper Table 1](#6.-Paper-Table-1)
7. [Paper Table 2](#7.-Paper-Table-2)
8. [Paper Figure 1 Dataset](#8.-Paper-Figure-1-Dataset)
9. [Paper Table 4](#9.-Paper-Table-4)
10. [Paper Table 5](#10.-Paper-Table-5)
11. [Paper Table 6](#11.-Paper-Table-6)
12. [Paper Table 7 / Figure 2](#12.-Paper-Table-7-/-Figure-2)
13. [Paper Figure 3](#13.-Paper-Figure-3)
14. [Validation Summary](#14.-Validation-Summary)
15. [Optional CSV Export Helpers](#15.-Optional-CSV-Export-Helpers)


## 1. Environment and Imports

Run this first. It checks the working directory, imports the project module, and shows package versions when available.


In [1]:
from pathlib import Path
import sys
import platform
import importlib

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Project root:', PROJECT_ROOT)

for pkg in ['numpy', 'pandas', 'scipy', 'pyfeng', 'pytest']:
    mod = importlib.import_module(pkg)
    print(f'{pkg}:', getattr(mod, '__version__', 'version unavailable'))


/Users/winfredzhao/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/winfredzhao/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Python: 3.9.13 (main, Aug 25 2022, 18:29:29) 
[Clang 12.0.0 ]
Platform: macOS-10.16-x86_64-i386-64bit
Project root: /Users/winfredzhao/Desktop/numerical-project
numpy: 1.24.4
pandas: 2.3.1
scipy: 1.9.1
pyfeng: version unavailable
pytest: 7.1.2


In [2]:
from sabr_replicate import (
    FDMConfig,
    MonteCarloConfig,
    SABRParams,
    case_table_3,
    conditional_integrated_variance_moments,
    european_call_price,
    fdm_benchmark_prices,
    figure1_moment_comparison,
    figure2_runtime_tradeoff,
    finite_difference_call_price,
    martingale_test,
    raw_moments_to_central_stats,
    run_figure3_experiment,
    run_full_validation,
    run_table1_experiment,
    run_table2_experiment,
    run_table4_experiment,
    run_table5_experiment,
    run_table6_experiment,
    run_table7_experiment,
    sample_conditional_integrated_variance,
    simulate_terminal_forward,
    simulate_terminal_forward_islah,
)
import pyfeng as pf


## 2. Project API Overview

This section is only for orientation. It lists the main functions used later in the notebook.


## 3. Core Mathematical Formulas

This section collects the formulas that the implementation is reproducing. In the code, the same objects appear as `SABRParams`, `MonteCarloConfig`, `sample_conditional_integrated_variance`, `sample_cev_exact`, and `simulate_terminal_forward`.

### 3.1 SABR model

The SABR dynamics are

$$
dF_t = \sigma_t F_t^\beta\, dW_t,
\qquad
\frac{d\sigma_t}{\sigma_t} = \nu\, dZ_t,
\qquad
dW_t\,dZ_t = \rho\,dt .
$$

The exact volatility step over one interval of length $h$ is

$$
\sigma_{t+h}
=
\sigma_t
\exp\left(\nu\sqrt{h}X_\sigma - \frac{1}{2}\nu^2 h\right),
\qquad X_\sigma \sim N(0,1).
$$

We use

$$
\hat\nu = \nu\sqrt{h},
\qquad
\beta^* = 1-\beta,
\qquad
\rho^* = \sqrt{1-\rho^2}.
$$

### 3.2 Algorithm 1: conditional average variance

After sampling $\sigma_{t+h}$, the first hard quantity is the normalized conditional average variance

$$
I_t^h
=
\frac{1}{\sigma_t^2 h}
\int_t^{t+h} \sigma_s^2\,ds
\;\Bigg|\;\sigma_{t+h}.
$$

Let

$$
\mu = \mathbb{E}\left[I_t^h \mid \sigma_{t+h}\right],
\qquad
v =
\frac{
\sqrt{\mathrm{Var}\left(I_t^h \mid \sigma_{t+h}\right)}
}{
\mathbb{E}\left[I_t^h \mid \sigma_{t+h}\right]
}.
$$

With fixed shift

$$
\lambda = \frac{5}{6},
$$

the lognormal shape parameter is

$$
a
=
\sqrt{\log\left(1 + \frac{v^2}{\lambda^2}\right)}
=
\sqrt{\log\left(1 + \frac{36}{25}v^2\right)}.
$$

Algorithm 1 samples

$$
I_t^h
\approx
\mu
\left[
\frac{1}{6}
+
\frac{5}{6}
\exp\left(aX - \frac{1}{2}a^2\right)
\right],
\qquad X\sim N(0,1).
$$

In the implementation, `conditional_integrated_variance_moments` supplies the moments and `sample_conditional_integrated_variance` performs this shifted-lognormal sampling.

### 3.3 Algorithm 2: martingale-preserving CEV approximation

For $0<\beta<1$, conditional on $\sigma_{t+h}$ and $I_t^h$,

$$
F_{t+h}\mid \sigma_{t+h}, I_t^h
\approx
\mathrm{CEV}_\beta\left(\bar F_t^h, (\rho^*)^2\sigma_t^2 h I_t^h\right).
$$

The conditional mean is

$$
\bar F_t^h
\approx
F_t
\exp\left(
\frac{\rho(\sigma_{t+h}-\sigma_t)}{\nu F_t^{\beta^*}}
-
\frac{\rho^2\sigma_t^2 h I_t^h}{2F_t^{2\beta^*}}
\right).
$$

This construction is meant to preserve the martingale condition:

$$
\mathbb{E}[F_{t+h}\mid \sigma_{t+h}, I_t^h] = \bar F_t^h,
\qquad
\mathbb{E}[F_{t+h}] = F_t.
$$

For the lognormal special case $\beta=1$,

$$
F_{t+h}
=
\bar F_t^h
\exp\left(
\rho^*\sigma_t\sqrt{hI_t^h}X
-
\frac{1}{2}(\rho^*)^2\sigma_t^2 hI_t^h
\right),
\qquad X\sim N(0,1).
$$

### 3.4 Algorithm 3: exact CEV sampling

For
$$
F_T \sim \mathrm{CEV}_\beta(F_0, \sigma_0^2T),
$$

define
$$
\alpha = \frac{1}{2\beta^*},
\qquad
z_0 = \frac{F_0^{2\beta^*}}{(\beta^*)^2\sigma_0^2T}.
$$

Sample
$$
X \sim \Gamma(\alpha,1),
$$
where $\Gamma(\alpha,1)$ denotes a Gamma distribution with shape $\alpha$ and unit scale.

If $X \ge z_0/2$, set $F_T = 0$ (absorbing boundary). Otherwise sample
$$
N \sim \mathrm{Poisson}(z_0/2 - X),
\qquad
z_T \sim 2\Gamma(N+1,1),
$$

and return
$$
F_T = \left((\beta^*)^2\sigma_0^2Tz_T\right)^{1/(2\beta^*)}.
$$

This provides an exact sampling scheme for the CEV distribution via a Poisson–Gamma mixture representation.

Inside SABR, this sampler is applied with
$$
F_0 \leftarrow \bar F_t^h,
\qquad
\sigma_0^2T \leftarrow (\rho^*)^2\sigma_t^2hI_t^h.
$$

### 3.5 Algorithm 4: full simulation over a time grid

For each time step:

1. Sample $\sigma_{t+h}$ exactly.
2. Sample $I_t^h$ using Algorithm 1.
3. Compute $\bar F_t^h$ using Algorithm 2.
4. Sample $F_{t+h}$ using Algorithm 3.
5. Repeat until maturity $T$.

The Monte Carlo European call estimator is

$$
\widehat C(K)
=
\frac{1}{N}\sum_{i=1}^N \max(F_T^{(i)}-K,0).
$$

The reported errors are

$$
\text{bias} = \widehat C - C_{\text{benchmark}},
\qquad
\text{relative error} = \frac{\widehat C-C_{\text{benchmark}}}{C_{\text{benchmark}}},
$$

$$
\text{RMS error} = \sqrt{\text{bias}^2 + \text{stdev}^2}.
$$

The martingale check verifies numerically that

$$
\mathbb{E}[F_T] = F_0.
$$


In [3]:
api_overview = pd.DataFrame(
    [
        ('simulate_terminal_forward', 'Main SABR Monte Carlo terminal simulator using the paper scheme'),
        ('simulate_terminal_forward_islah', 'Appendix B / Islah-style comparison branch'),
        ('sample_conditional_integrated_variance', 'Algorithm 1 style conditional average-variance sampling'),
        ('conditional_integrated_variance_moments', 'Conditional raw moments of normalized average variance'),
        ('run_table1_experiment', 'Paper Table 1 dataset'),
        ('run_table2_experiment', 'Paper Table 2 dataset'),
        ('run_table4_experiment', 'Paper Table 4 dataset'),
        ('run_table5_experiment', 'Paper Table 5 dataset'),
        ('run_table6_experiment', 'Paper Table 6 dataset'),
        ('run_table7_experiment', 'Paper Table 7 / Figure 2 dataset'),
        ('run_figure3_experiment', 'Paper Figure 3 dataset'),
        ('run_full_validation', 'Repository validation harness'),
    ],
    columns=['function', 'purpose'],
)
api_overview


,function,purpose
0,simulate_terminal_forward,Main SABR Monte Carlo terminal simulator using...
1,simulate_terminal_forward_islah,Appendix B / Islah-style comparison branch
2,sample_conditional_integrated_variance,Algorithm 1 style conditional average-variance...
3,conditional_integrated_variance_moments,Conditional raw moments of normalized average ...
4,run_table1_experiment,Paper Table 1 dataset
5,run_table2_experiment,Paper Table 2 dataset
6,run_table4_experiment,Paper Table 4 dataset
7,run_table5_experiment,Paper Table 5 dataset
8,run_table6_experiment,Paper Table 6 dataset
9,run_table7_experiment,Paper Table 7 / Figure 2 dataset


## 4. Table 3 Cases

These are the paper parameter presets used throughout later sections.


In [4]:
case_df = pd.DataFrame(case_table_3()).T
case_df.index.name = 'case'
case_df


,f0,sigma0,nu,rho,beta,maturity
case,,,,,,
Case I,1.00,0.25,0.3,-0.8,0.3,10.0
Case II,1.00,0.25,0.3,-0.5,0.6,10.0
Case III,0.05,0.40,0.6,0.0,0.3,1.0
Case IV,1.10,0.40,0.8,-0.3,0.3,4.0
Case V,1.10,0.30,0.5,-0.8,0.4,10.0


## 5. Sanity Checks

These cells verify model-level behavior that should hold independently of the paper tables.


### 5.1 `nu = 0` should reduce SABR to CEV


In [5]:
params = SABRParams(f0=1.0, sigma0=0.25, nu=0.0, rho=-0.4, beta=0.3)
mc = MonteCarloConfig(maturity=1.0, step=1.0, n_paths=200_000, seed=12345)
strikes = np.array([0.6, 1.0, 1.4])

terminal = simulate_terminal_forward(params, mc)
mc_prices = np.array([european_call_price(terminal, k) for k in strikes])
cev_prices = pf.Cev(sigma=params.sigma0, beta=params.beta, is_fwd=True).price(strikes, params.f0, mc.maturity)

pd.DataFrame({
    'strike': strikes,
    'mc_price': mc_prices,
    'cev_price': cev_prices,
    'error': mc_prices - cev_prices,
})


,strike,mc_price,cev_price,error
0,0.6,0.404583,0.404036,0.000547
1,1.0,0.099984,0.099602,0.000382
2,1.4,0.007369,0.007385,-0.000016


### 5.2 `beta = 1, nu = 0` should reduce SABR to Black-Scholes / lognormal


In [6]:
params = SABRParams(f0=1.0, sigma0=0.2, nu=0.0, rho=-0.75, beta=1.0)
mc = MonteCarloConfig(maturity=1.0, step=1.0, n_paths=200_000, seed=12345)
strikes = np.array([0.8, 1.0, 1.2])

terminal = simulate_terminal_forward(params, mc)
mc_prices = np.array([european_call_price(terminal, k) for k in strikes])
bsm_prices = pf.Bsm(sigma=params.sigma0, is_fwd=True).price(strikes, params.f0, mc.maturity)

pd.DataFrame({
    'strike': strikes,
    'mc_price': mc_prices,
    'bsm_price': bsm_prices,
    'error': mc_prices - bsm_prices,
})


,strike,mc_price,bsm_price,error
0,0.8,0.212078,0.211859,0.000219
1,1.0,0.079613,0.079656,-0.000043
2,1.2,0.021431,0.021473,-0.000042


### 5.3 Conditional average-variance moment spot check

This is a small diagnostic cell for the `I_t^h` machinery used inside the full simulator.


In [7]:
sigma_t = np.array([0.2])
sigma_next = np.array([0.24])
nu = 0.4
h = 1.0

mu1, mu2_raw, mu3_raw, mu4_raw = conditional_integrated_variance_moments(sigma_t, sigma_next, nu, h)
mean, var, std, cv, skewness, ex_kurtosis = raw_moments_to_central_stats(mu1, mu2_raw, mu3_raw, mu4_raw)

pd.DataFrame({
    'mu1': mu1,
    'mu2_raw': mu2_raw,
    'mu3_raw': mu3_raw,
    'mu4_raw': mu4_raw,
    'var': var,
    'std': std,
    'cv': cv,
    'skewness': skewness,
    'ex_kurtosis': ex_kurtosis,
})


,mu1,mu2_raw,mu3_raw,mu4_raw,var,std,cv,skewness,ex_kurtosis
0,1.272973,1.712454,2.438791,3.683759,0.091994,0.303305,0.238265,0.884234,1.467284


### 5.4 Martingale sanity check


In [8]:
case_v = case_table_3()['Case V']
params = SABRParams(
    f0=case_v['f0'],
    sigma0=case_v['sigma0'],
    nu=case_v['nu'],
    rho=case_v['rho'],
    beta=case_v['beta'],
)

martingale_test(params, maturities=[1, 2, 4, 6, 8, 10], step=1.0, n_paths=30_000, seed0=101)


,maturity,mean_terminal,std_terminal,stderr_terminal,martingale_error,z_score,runtime_sec,step,n_paths,conclusion
0,1.0,1.098478,0.305621,0.001765,-0.001522,-0.862636,0.043611,1.0,30000,Monte Carlo noise dominated
1,2.0,1.102482,0.422185,0.002437,0.002482,1.018128,0.043501,1.0,30000,Monte Carlo noise dominated
2,4.0,1.107439,0.563293,0.003252,0.007439,2.287301,0.070922,1.0,30000,Monte Carlo noise dominated
3,6.0,1.105598,0.643586,0.003716,0.005598,1.506589,0.105387,1.0,30000,Monte Carlo noise dominated
4,8.0,1.101523,0.703496,0.004062,0.001523,0.374890,0.132791,1.0,30000,Monte Carlo noise dominated
5,10.0,1.099555,0.741081,0.004279,-0.000445,-0.103937,0.172427,1.0,30000,Monte Carlo noise dominated


### 5.5 `|rho| = 1` Islah edge case stability


In [9]:
rows = []
for beta in (0.4, 0.6, 0.8):
    params = SABRParams(f0=1.0, sigma0=0.2, nu=0.2, rho=1.0, beta=beta)
    mc = MonteCarloConfig(maturity=1.0, step=1.0, n_paths=5_000, seed=123)
    terminal = simulate_terminal_forward_islah(params, mc)
    rows.append({
        'beta': beta,
        'all_finite': bool(np.isfinite(terminal).all()),
        'all_nonnegative': bool((terminal >= 0.0).all()),
        'mean_terminal': float(np.mean(terminal)),
        'min_terminal': float(np.min(terminal)),
    })

pd.DataFrame(rows)


,beta,all_finite,all_nonnegative,mean_terminal,min_terminal
0,0.4,True,True,1.009120,0.536459
1,0.6,True,True,1.013306,0.558478
2,0.8,True,True,1.017677,0.577711


## 6. Paper Table 1

By default, this uses the paper table benchmarks embedded in the repo. To compare against the repo PDE benchmark instead, run the later FDM cells or pass benchmark providers in standalone scripts.


In [10]:
table1_df = run_table1_experiment(n_paths=100_000, n_repeats=50, seed0=12345)
table1_df


,strike,mean_price,stdev_price,mean_terminal,stdev_terminal_estimator,stderr_price,stderr_terminal_estimator,n_repeats,n_paths,step,maturity,runtime_sec_mean,benchmark_price,bias,relative_error,rho,nu,beta
0,1.0,0.079139,0.000350,1.000117,0.000676,0.000049,0.000096,50.0,100000.0,1.00,1.0,0.039041,0.07910,0.000039,0.000497,-0.75,0.2,1.0
1,1.0,0.079153,0.000368,1.000035,0.000651,0.000052,0.000092,50.0,100000.0,0.50,1.0,0.074864,0.07910,0.000053,0.000668,-0.75,0.2,1.0
2,1.0,0.079105,0.000403,1.000101,0.000647,0.000057,0.000091,50.0,100000.0,0.25,1.0,0.149698,0.07910,0.000005,0.000067,-0.75,0.2,1.0
3,1.0,0.079425,0.000334,0.999981,0.000583,0.000047,0.000082,50.0,100000.0,1.00,1.0,0.037206,0.07942,0.000005,0.000057,-0.50,0.2,1.0
4,1.0,0.079858,0.000353,1.000222,0.000623,0.000050,0.000088,50.0,100000.0,1.00,1.0,0.037266,0.07969,0.000168,0.002107,-0.25,0.2,1.0
5,1.0,0.078509,0.000340,0.999892,0.000602,0.000048,0.000085,50.0,100000.0,1.00,1.0,0.037164,0.07860,-0.000091,-0.001152,-0.75,0.4,1.0
6,1.0,0.078056,0.000325,0.999862,0.000635,0.000046,0.000090,50.0,100000.0,1.00,1.0,0.037961,0.07811,-0.000054,-0.000694,-0.75,0.6,1.0


## 7. Paper Table 2


In [11]:
table2_df = run_table2_experiment(n_paths=100_000, n_repeats=50, seed0=12345)
table2_df


,strike,mean_price,stdev_price,mean_terminal,stdev_terminal_estimator,stderr_price,stderr_terminal_estimator,n_repeats,n_paths,step,maturity,runtime_sec_mean,benchmark_price,bias,relative_error,rho,nu,beta
0,1.0,0.080272,0.000465,0.999916,0.000687,0.000066,0.000097,50.0,100000.0,1.00,1.0,0.040747,0.07989,0.000382,0.004786,1.00,0.2,0.4
1,1.0,0.080308,0.000564,0.999972,0.000845,0.000080,0.000119,50.0,100000.0,1.00,1.0,0.036682,0.08002,0.000288,0.003605,1.00,0.2,0.6
2,1.0,0.080416,0.000479,1.000109,0.000634,0.000068,0.000090,50.0,100000.0,1.00,1.0,0.036116,0.08017,0.000246,0.003071,1.00,0.2,0.8
3,1.0,0.080746,0.000626,0.999897,0.000775,0.000089,0.000110,50.0,100000.0,1.00,1.0,0.036602,0.08044,0.000306,0.003809,1.00,0.4,0.8
4,1.0,0.080668,0.001318,0.999282,0.001365,0.000186,0.000193,50.0,100000.0,1.00,1.0,0.036195,0.08043,0.000238,0.002958,1.00,0.8,0.8
5,1.0,0.080169,0.000431,0.999867,0.000646,0.000061,0.000091,50.0,100000.0,0.50,1.0,0.088552,0.07989,0.000279,0.003489,1.00,0.2,0.4
6,1.0,0.080356,0.000428,1.000160,0.000645,0.000061,0.000091,50.0,100000.0,0.50,1.0,0.081516,0.08002,0.000336,0.004204,1.00,0.2,0.6
7,1.0,0.080461,0.000555,1.000217,0.000782,0.000079,0.000111,50.0,100000.0,0.50,1.0,0.073495,0.08017,0.000291,0.003629,1.00,0.2,0.8
8,1.0,0.080583,0.000517,0.999917,0.000736,0.000073,0.000104,50.0,100000.0,0.50,1.0,0.081843,0.08044,0.000143,0.001780,1.00,0.4,0.8
9,1.0,0.081096,0.002270,1.000229,0.002316,0.000321,0.000328,50.0,100000.0,0.50,1.0,0.079053,0.08043,0.000666,0.008275,1.00,0.8,0.8


## 8. Paper Figure 1 Dataset

This generates the skewness and ex-kurtosis comparison data used for the figure. Plotting is optional; the dataset itself is enough for validation.


In [ ]:
figure1_df = figure1_moment_comparison(hat_nu=0.4)
figure1_df.head(12)


In [ ]:
ax = figure1_df.plot(x='z_hat', y=['exact_skewness', 'ln_skewness', 'sln_fixed_skewness', 'sln_exact_skewness'], figsize=(10, 4), title='Figure 1 style skewness comparison')
ax.set_ylabel('skewness')


## 9. Paper Table 4

Case I. This includes the simulated rows and the available analytic comparison rows.


In [ ]:
table4_df = run_table4_experiment(n_paths=100_000, n_repeats=50, seed0=12345)
table4_df


## 10. Paper Table 5

Case II.


In [ ]:
table5_df = run_table5_experiment(n_paths=100_000, n_repeats=50, seed0=12345)
table5_df


## 11. Paper Table 6

Case III. This is the runtime / bias comparison against paper-reference baseline rows.


In [ ]:
table6_df = run_table6_experiment(n_paths=100_000, n_repeats=50, seed0=12345)
table6_df


## 12. Paper Table 7 / Figure 2

This section can be slow. It reproduces the convergence and runtime trade-off dataset.


In [ ]:
table7_df = run_table7_experiment(
    n_paths_base=160_000,
    n_repeats=5,
    seed0=12345,
    benchmark_source='mc',
)
table7_df


In [ ]:
figure2_df = figure2_runtime_tradeoff(
    n_paths_base=160_000,
    n_repeats=5,
    seed0=12345,
    benchmark_source='mc',
)
figure2_df


## 13. Paper Figure 3

This compares the paper scheme against the Islah approximation across maturities.


In [ ]:
figure3_df = run_figure3_experiment(
    n_paths=100_000,
    n_repeats=2,
    seed0=12345,
    benchmark_source='mc',
)
figure3_df.head(20)


In [ ]:
pivot_forward = figure3_df.pivot_table(index='maturity', columns=['model', 'step'], values='forward_error')
pivot_option = figure3_df.pivot_table(index='maturity', columns=['model', 'step'], values='option_error')

display(pivot_forward)
display(pivot_option)


## 14. Validation Summary

Start with quick mode. The paper-scale run is much slower.


In [ ]:
validation_quick = run_full_validation(quick_mode=True)
pd.Series({
    'table1_status': validation_quick['table1_status'],
    'table2_status': validation_quick['table2_status'],
    'overall_conclusion': validation_quick['overall_conclusion'],
    'replication_conclusion': validation_quick['replication_conclusion'],
})


In [ ]:
# Uncomment this only when you want the full validation sweep.
# validation_full = run_full_validation(quick_mode=False)
# pd.Series({
#     'table1_status': validation_full['table1_status'],
#     'table2_status': validation_full['table2_status'],
#     'overall_conclusion': validation_full['overall_conclusion'],
#     'replication_conclusion': validation_full['replication_conclusion'],
# })


## 15. Optional CSV Export Helpers

Use these after running the sections you care about.


In [ ]:
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'notebook_exports'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR


In [ ]:
# Example exports. Uncomment what you need.
# table1_df.to_csv(OUTPUT_DIR / 'table1.csv', index=False)
# table2_df.to_csv(OUTPUT_DIR / 'table2.csv', index=False)
# table4_df.to_csv(OUTPUT_DIR / 'table4.csv', index=False)
# table5_df.to_csv(OUTPUT_DIR / 'table5.csv', index=False)
# table6_df.to_csv(OUTPUT_DIR / 'table6.csv', index=False)
# table7_df.to_csv(OUTPUT_DIR / 'table7.csv', index=False)
# figure1_df.to_csv(OUTPUT_DIR / 'figure1_dataset.csv', index=False)
# figure3_df.to_csv(OUTPUT_DIR / 'figure3_dataset.csv', index=False)
